# 📊 Macro-Conditional & Regime-Dependent Credit Risk Modeling
## MScFE Capstone — Group 14074

- **Taoufiq OUEDRAOGO**
- **Wickliff Siriga OGARI**

---


$$ PD = Pr(\text{default} | X_\text{borrower}, M_t, R_t) $$


| Symbol | Meaning |
|--------|---------|
| $X_\text{borrower}$ | Borrower-specific characteristics (FICO, DTI, income …) |
| $M_t$ | Macroeconomic variables at loan issuance time *t* |
| $R_t$ | Latent market regime at time *t* (detected via HMM / GMM) |

<br>


3-tier model hierarchy:
- **Model 1 - Baseline** : $PD = f(X_\text{borrower})$
- **Model 2 - Macro-Augmented** : $PD = f(X_\text{borrower}, M_t)$
- **Model 3 - Regime-Dependent** : $PD = f(X_\text{borrower}, M_t, R_t)$


In [ ]:
import warnings, os
warnings.filterwarnings("ignore")

import ssl
ssl._create_default_https_context = ssl._create_unverified_context

import numpy as np
import pandas as pd

import kagglehub

import seaborn as sns
import matplotlib.pyplot as plt

# ── ML / Stats ────────────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score, roc_curve, average_precision_score,
    precision_recall_curve, brier_score_loss,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.calibration import calibration_curve, CalibratedClassifierCV
from sklearn.mixture import GaussianMixture
from sklearn.cluster import KMeans, AgglomerativeClustering

# ── XGBoost / LightGBM ────────────────────────────────────────────────────
#from xgboost import XGBClassifier
#import lightgbm as lgb


# ── HMM ───────────────────────────────────────────────────────────────────
#from hmmlearn import hmm

# ── Misc ──────────────────────────────────────────────────────────────────
from scipy import stats
from datetime import datetime
#import shap

# ── Reproducibility ───────────────────────────────────────────────────────
SEED = 42
np.random.seed(SEED)

PALETTE = {"Base": "#4878d0", "Macro-Augmented": "#ee854a", "Regime-Dependent": "#6acc65"}

---

## 1. Data Pipeline - Objective O1

**Point-in-Time join** ensures no look-ahead bias:

$$D_{merged} = \{(x_{i,t}, m_\tau)\;|\;\tau = \max(t_{macro} \leq t_{loan})\}$$

> - Lending Club public dataset [from Kaggle](https://www.kaggle.com/datasets/adarshsng/lending-club-loan-data-csv?select=loan.csv)
> - Macro data [from FRED API](https://fred.stlouisfed.org/) 


### Load macro data

In [2]:
def load_macro_data():
    """
    Load real macroeconomic data from FRED  
    """
    import ssl
    ssl._create_default_https_context = ssl._create_unverified_context

    def get_fred_series(series_id):
        url = f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={series_id}"
        df = pd.read_csv(url)
        df.columns = ["date", series_id]
        df["date"] = pd.to_datetime(df["date"])
        return df.set_index("date")

    # Retrieve key macro series
    fed = get_fred_series("DFF")        # Fed Funds Rate
    unemp = get_fred_series("UNRATE")     # Unemployment rate
    cpi = get_fred_series("CPIAUCSL")   # Inflation index
    ind_prod = get_fred_series("INDPRO")     # Industrial production
    gdp = get_fred_series("GDP")        # GDP (quarterly)
    hy_spread = get_fred_series("BAMLH0A0HYM2")  # High yield spread

    # Merge all
    macro = fed.join([unemp, cpi, ind_prod, gdp, hy_spread], how="outer")

    # Rename columns for consistency
    macro.columns = [
        "fed_funds",
        "unemp_rate",
        "cpi",
        "ind_prod",
        "real_gdp",
        "hy_spread"
    ]

    # Sort and clean
    macro = macro.sort_index()

    # Forward-fill missing values (critical for macro data)
    macro = macro.ffill()
    return macro.reset_index()

In [3]:
macro_df = load_macro_data()
macro_df.head()

,date,fed_funds,unemp_rate,cpi,ind_prod,real_gdp,hy_spread
0,1919-01-01,NaN,NaN,NaN,4.8739,NaN,NaN
1,1919-02-01,NaN,NaN,NaN,4.6585,NaN,NaN
2,1919-03-01,NaN,NaN,NaN,4.5238,NaN,NaN
3,1919-04-01,NaN,NaN,NaN,4.6046,NaN,NaN
4,1919-05-01,NaN,NaN,NaN,4.6315,NaN,NaN


### Load loan data

In [4]:
# Download latest version from kaggle
path = kagglehub.dataset_download("adarshsng/lending-club-loan-data-csv")

In [5]:
print("Path to dataset files:", path)

Path to dataset files: /Users/taoufiq/.cache/kagglehub/datasets/adarshsng/lending-club-loan-data-csv/versions/1


In [6]:
print(os.listdir(path))

['loan.csv', 'LCDataDictionary.xlsx']


In [37]:
KEEP_COLS = [
    "issue_d", "loan_amnt", "term", "int_rate", "grade", "sub_grade",
    "emp_length", "home_ownership", "annual_inc", "dti", "delinq_2yrs",
    #"fico_range_low", "fico_range_high", 
    "inq_last_6mths",
    "mths_since_last_delinq", "pub_rec_bankruptcies", "num_accts_ever_120_pd",
    "pct_tl_nvr_dlq", "loan_status", "earliest_cr_line",
]

file_path = os.path.join(path, "loan.csv")
loan_df = pd.read_csv(file_path, low_memory=False, nrows=1000)[KEEP_COLS]
print(loan_df.shape)
loan_df.head()

(1000, 18)


,issue_d,loan_amnt,term,int_rate,grade,sub_grade,emp_length,home_ownership,annual_inc,dti,delinq_2yrs,inq_last_6mths,mths_since_last_delinq,pub_rec_bankruptcies,num_accts_ever_120_pd,pct_tl_nvr_dlq,loan_status,earliest_cr_line
0,Dec-2018,2500,36 months,13.56,C,C1,10+ years,RENT,55000,18.24,0,1,NaN,1,0,100.0,Current,Apr-2001
1,Dec-2018,30000,60 months,18.94,D,D2,10+ years,MORTGAGE,90000,26.52,0,0,71.0,1,0,95.0,Current,Jun-1987
2,Dec-2018,5000,36 months,17.97,D,D1,6 years,MORTGAGE,59280,10.51,0,0,NaN,0,0,100.0,Current,Apr-2011
3,Dec-2018,4000,36 months,18.94,D,D2,10+ years,MORTGAGE,92000,16.74,0,0,NaN,0,0,100.0,Current,Feb-2006
4,Dec-2018,30000,60 months,16.14,C,C4,10+ years,MORTGAGE,57250,26.35,0,0,NaN,0,0,92.3,Current,Dec-2000


In [35]:
# Basic cleaning
loan_df['issue_d'] = pd.to_datetime(loan_df['issue_d'], errors='coerce')
loan_df = loan_df.sort_values("issue_d").reset_index(drop=True)

loan_df["term"] = loan_df["term"].str.extract(r"(\d+)").astype(int)
loan_df["emp_length"] = loan_df["emp_length"].str.extract(r"(\d+)").astype(float).fillna(0)
loan_df["earliest_credit_line"] = pd.to_datetime(loan_df["earliest_cr_line"], format="%b-%Y").dt.year
loan_df.head()

,issue_d,loan_amnt,term,int_rate,grade,sub_grade,emp_length,home_ownership,annual_inc,dti,delinq_2yrs,inq_last_6mths,mths_since_last_delinq,pub_rec_bankruptcies,num_accts_ever_120_pd,pct_tl_nvr_dlq,loan_status,earliest_cr_line,earliest_credit_line
0,2018-12-01,2500,36,13.56,C,C1,10.0,RENT,55000,18.24,0,1,NaN,1,0,100.0,Current,Apr-2001,2001
1,2018-12-01,10000,36,6.46,A,A1,10.0,MORTGAGE,115000,10.41,2,1,18.0,0,0,83.3,Current,Feb-2007,2007
2,2018-12-01,30000,36,16.14,C,C4,1.0,MORTGAGE,90000,7.60,0,0,NaN,0,0,100.0,Current,Apr-1991,1991
3,2018-12-01,13000,36,8.19,A,A4,10.0,MORTGAGE,35000,22.57,0,0,69.0,0,0,96.3,Current,Feb-2004,2004
4,2018-12-01,24000,60,10.72,B,B2,10.0,MORTGAGE,76500,18.51,0,0,NaN,0,0,100.0,Current,Jul-2005,2005


In [ ]:
# Define target
loan_df = loan_df[loan_df["loan_status"].isin(["Fully Paid", "Charged Off"])]
loan_df["loan_default"] = (loan_df["loan_status"] == "Charged Off").astype(int)

loan_df.head()

,issue_d,loan_amnt,term,int_rate,grade,sub_grade,emp_length,home_ownership,annual_inc,dti,delinq_2yrs,inq_last_6mths,mths_since_last_delinq,pub_rec_bankruptcies,num_accts_ever_120_pd,pct_tl_nvr_dlq,loan_status,earliest_cr_line,earliest_credit_line,default
51,2018-12-01,10450,60,12.98,B,B5,10.0,MORTGAGE,58000,21.17,0,1,NaN,0,0,100.0,Fully Paid,May-2003,2003,0
165,2018-12-01,5000,36,7.56,A,A3,5.0,MORTGAGE,98000,14.78,0,0,59.0,0,1,92.3,Fully Paid,Mar-2008,2008,0
231,2018-12-01,12950,36,7.56,A,A3,10.0,MORTGAGE,55000,19.88,0,1,NaN,0,0,100.0,Fully Paid,Dec-2005,2005,0
273,2018-12-01,10000,36,11.80,B,B4,1.0,RENT,100000,10.60,1,1,6.0,0,7,61.9,Fully Paid,Jun-1982,1982,0
375,2018-12-01,3500,36,18.94,D,D2,10.0,MORTGAGE,127300,17.91,2,2,3.0,0,0,93.0,Fully Paid,Sep-2001,2001,0


In [25]:
def pit_join(loan_df, macro_df):
    """Merge macro onto loans using the last available macro observation
    at or before each loan's issue date — prevents data leakage."""
    merged = pd.merge_asof(
        loan_df.sort_values("issue_d"),
        macro_df.sort_values("date"),
        left_on="issue_d", right_on="date",
        direction="backward"
    ).drop(columns="date")
    return merged


data = pit_join(loan_df, macro_df)
data.head()

,issue_d,loan_amnt,term,int_rate,grade,sub_grade,emp_length,home_ownership,annual_inc,dti,...,pct_tl_nvr_dlq,loan_status,earliest_cr_line,earliest_credit_line,fed_funds,unemp_rate,cpi,ind_prod,real_gdp,hy_spread
0,2018-12-01,2500,36,13.56,C,C1,10.0,RENT,55000,18.24,...,100.0,Current,Apr-2001,2001,2.2,3.9,252.767,104.0936,20917.867,NaN
1,2018-12-01,30000,36,16.14,C,C4,1.0,MORTGAGE,90000,7.60,...,100.0,Current,Apr-1991,1991,2.2,3.9,252.767,104.0936,20917.867,NaN
2,2018-12-01,13000,36,8.19,A,A4,10.0,MORTGAGE,35000,22.57,...,96.3,Current,Feb-2004,2004,2.2,3.9,252.767,104.0936,20917.867,NaN
3,2018-12-01,24000,60,10.72,B,B2,10.0,MORTGAGE,76500,18.51,...,100.0,Current,Jul-2005,2005,2.2,3.9,252.767,104.0936,20917.867,NaN
4,2018-12-01,6000,36,7.02,A,A2,10.0,MORTGAGE,20000,57.14,...,95.2,Current,Jul-2006,2006,2.2,3.9,252.767,104.0936,20917.867,NaN


In [27]:
def engineer_features(df):
    """Adds derived features used by both borrower and macro models."""
    df = df.copy()
    # Borrower
    df["credit_age"] = df["issue_d"].dt.year - df["earliest_credit_line"]
    df["high_dti"] = (df["dti"] > 30).astype(int)
    
    # Macro deltas (month-over-month changes - computed on sorted data)
    for col in ["fed_funds","unemp_rate","hy_spread","cpi"]:
        df[f"{col}_delta"] = df[col].diff().fillna(0)

    # Macro composite stress index
    df["macro_stress"] = (
        df["unemp_rate"] / df["unemp_rate"].max() * 0.4 +
        df["hy_spread"]  / df["hy_spread"].max()  * 0.4 +
        (1 - df["fed_funds"] / df["fed_funds"].max()) * 0.2
    )
    return df


data = engineer_features(data)
data.head()

,issue_d,loan_amnt,term,int_rate,grade,sub_grade,emp_length,home_ownership,annual_inc,dti,...,ind_prod,real_gdp,hy_spread,credit_age,high_dti,fed_funds_delta,unemp_rate_delta,hy_spread_delta,cpi_delta,macro_stress
0,2018-12-01,2500,36,13.56,C,C1,10.0,RENT,55000,18.24,...,104.0936,20917.867,NaN,17,0,0.0,0.0,0.0,0.0,NaN
1,2018-12-01,30000,36,16.14,C,C4,1.0,MORTGAGE,90000,7.60,...,104.0936,20917.867,NaN,27,0,0.0,0.0,0.0,0.0,NaN
2,2018-12-01,13000,36,8.19,A,A4,10.0,MORTGAGE,35000,22.57,...,104.0936,20917.867,NaN,14,0,0.0,0.0,0.0,0.0,NaN
3,2018-12-01,24000,60,10.72,B,B2,10.0,MORTGAGE,76500,18.51,...,104.0936,20917.867,NaN,13,0,0.0,0.0,0.0,0.0,NaN
4,2018-12-01,6000,36,7.02,A,A2,10.0,MORTGAGE,20000,57.14,...,104.0936,20917.867,NaN,12,1,0.0,0.0,0.0,0.0,NaN


In [29]:
def build_target(df, seed=SEED):
    """Synthetic default label driven by both micro & macro signals."""
    np.random.seed(seed)
    logit = (
        #(720 - df["fico"]) / 70
        + (df["unemp_rate"] - 5.5) / 2.5
        + (df["hy_spread"]  - 4.5) / 3
        + (df["dti"] - 20) / 25
        - df["annual_inc"].pipe(np.log) / 12
    )
    prob = 1 / (1 + np.exp(-logit))
    df["default"] = (prob > np.random.rand(len(df))).astype(int)
    return df


data = build_target(data)
data.head()

,issue_d,loan_amnt,term,int_rate,grade,sub_grade,emp_length,home_ownership,annual_inc,dti,...,real_gdp,hy_spread,credit_age,high_dti,fed_funds_delta,unemp_rate_delta,hy_spread_delta,cpi_delta,macro_stress,default
0,2018-12-01,2500,36,13.56,C,C1,10.0,RENT,55000,18.24,...,20917.867,NaN,17,0,0.0,0.0,0.0,0.0,NaN,0
1,2018-12-01,30000,36,16.14,C,C4,1.0,MORTGAGE,90000,7.60,...,20917.867,NaN,27,0,0.0,0.0,0.0,0.0,NaN,0
2,2018-12-01,13000,36,8.19,A,A4,10.0,MORTGAGE,35000,22.57,...,20917.867,NaN,14,0,0.0,0.0,0.0,0.0,NaN,0
3,2018-12-01,24000,60,10.72,B,B2,10.0,MORTGAGE,76500,18.51,...,20917.867,NaN,13,0,0.0,0.0,0.0,0.0,NaN,0
4,2018-12-01,6000,36,7.02,A,A2,10.0,MORTGAGE,20000,57.14,...,20917.867,NaN,12,1,0.0,0.0,0.0,0.0,NaN,0
